# Kaggle Runtime Benchmark

This notebook is ready to upload to Kaggle. It installs the required packages, measures a small GPT-2 training step on the available GPU(s), and estimates the runtime for a 2xT4 setup.

Run the cells from top to bottom. If Kaggle gives you 2 GPUs, the benchmark will use them automatically when possible.

In [7]:
!pip -q install torch transformers datasets accelerate sentencepiece textstat

In [8]:
import math
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        print(f"GPU {idx}: {torch.cuda.get_device_name(idx)}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [9]:
MODEL_NAME = "gpt2"
SEQ_LEN = 256
BATCH_SIZE = 4
WARMUP_STEPS = 3
BENCHMARK_STEPS = 10
EPOCHS = 2.0
NUM_EXAMPLES = 50000
AVG_TOKENS_PER_EXAMPLE = 180
MULTI_GPU_EFFICIENCY = 0.85

import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.train()

device_count = torch.cuda.device_count()
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
model = model.to(device)
if device_count > 1:
    model = torch.nn.DataParallel(model)

sample_text = ("A brave little fox explored the forest and helped a lost rabbit find home. " * 20)
batch_encoding = tokenizer(
    [sample_text] * BATCH_SIZE,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=SEQ_LEN,
)
input_ids = batch_encoding["input_ids"].to(device)
attention_mask = batch_encoding["attention_mask"].to(device)
labels = input_ids.clone()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

def train_step():
    optimizer.zero_grad(set_to_none=True)
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    if loss.ndim > 0:
        loss = loss.mean()
    loss.backward()
    optimizer.step()
    return float(loss.detach().cpu())

if use_cuda:
    torch.cuda.synchronize()
for _ in range(WARMUP_STEPS):
    _ = train_step()
if use_cuda:
    torch.cuda.synchronize()

start = time.perf_counter()
loss_values = []
for _ in range(BENCHMARK_STEPS):
    loss_values.append(train_step())
if use_cuda:
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start

measured_tokens = BENCHMARK_STEPS * BATCH_SIZE * SEQ_LEN
single_run_tps = measured_tokens / elapsed

print(f"Average loss: {sum(loss_values) / len(loss_values):.4f}")
print(f"Measured throughput: {single_run_tps:.2f} tokens/sec")
print(
    f"Estimated 1xT4 runtime for {NUM_EXAMPLES} examples: "
    f"{(NUM_EXAMPLES * AVG_TOKENS_PER_EXAMPLE * EPOCHS) / single_run_tps / 3600.0:.2f} hours"
)

estimated_two_gpu_tps = single_run_tps * 2 * MULTI_GPU_EFFICIENCY
estimated_two_gpu_hours = (NUM_EXAMPLES * AVG_TOKENS_PER_EXAMPLE * EPOCHS) / estimated_two_gpu_tps / 3600.0
print(f"Estimated 2xT4 throughput (@{MULTI_GPU_EFFICIENCY:.0%} efficiency): {estimated_two_gpu_tps:.2f} tokens/sec")
print(f"Estimated 2xT4 runtime: {estimated_two_gpu_hours:.2f} hours")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Average loss: 0.1712
Measured throughput: 3772.15 tokens/sec
Estimated 1xT4 runtime for 50000 examples: 1.33 hours
Estimated 2xT4 throughput (@85% efficiency): 6412.65 tokens/sec
Estimated 2xT4 runtime: 0.78 hours


## Notes

- If Kaggle exposes 2 GPUs, the notebook uses `DataParallel` automatically.
- If only 1 GPU is available, the notebook still runs and estimates the 2xT4 time by scaling the measured throughput.
- For your report, mention the efficiency assumption used for the 2xT4 estimate.